## 1. IMPORTS

In [41]:
import pandas as pd
import numpy as np
import xgboost as xgb
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_curve, auc
from imblearn.over_sampling import SMOTE

## 2. LOAD & SHUFFLE DATA

In [39]:
print("\n=================LOADING & SHUFFLING DATA =================")
start_time = time.time()

df = pd.read_csv('sgcc_engineered_features.csv', index_col='CONS_NO')
print(f"Total Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\nOriginal FLAG distribution:\n", df['FLAG'].value_counts())

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("Data shuffled successfully")


=================LOADING & SHUFFLING DATA =================
Total Rows: 42372, Columns: 9

Original FLAG distribution:
 FLAG
0    38757
1     3615
Name: count, dtype: int64
Data shuffled successfully


## 3. FEATURES & TARGET

In [40]:
X = df.drop('FLAG', axis=1)
y = df['FLAG']
print(f"\nFeatures shape: {X.shape}, Target shape: {y.shape}")



Features shape: (42372, 8), Target shape: (42372,)


## 4. TRAIN / TEST SPLIT (STRATIFIED)

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("\nTRAIN SET distribution:\n", y_train.value_counts())
print("\nTEST SET distribution:\n", y_test.value_counts())



TRAIN SET distribution:
 FLAG
0    29068
1     2711
Name: count, dtype: int64

TEST SET distribution:
 FLAG
0    9689
1     904
Name: count, dtype: int64


## 5. HANDLE IMBALANCE WITH SMOTE

In [45]:
print("\n=================SMOTE OVERSAMPLING =================")
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print(" SMOTE applied")
print("Resampled TRAIN distribution:\n", pd.Series(y_train_res).value_counts())



=================SMOTE OVERSAMPLING =================
 SMOTE applied
Resampled TRAIN distribution:
 FLAG
0    29068
1    29068
Name: count, dtype: int64


## 6. BASE XGBOOST MODEL

In [46]:
print("\n=================INITIALIZE BASE MODEL=================")
xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)
print("Base model ready")


=================INITIALIZE BASE MODEL=================
Base model ready


## 7. HYPERPARAMETER GRID SEARCH + CV 

In [7]:
print("\n================= STEP 6: GRID SEARCH + CV =================")
param_grid = {
    'n_estimators': [400, 600],
    'max_depth': [5, 7, 9],
    'learning_rate': [0.03, 0.05],
    'subsample': [0.7, 0.8],
    'colsample_bytree': [0.7, 0.8],
    'gamma': [0.1, 0.2, 0.3],
    'min_child_weight': [1, 3, 5]
}

grid = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='recall',
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train_res, y_train_res)
best_model = grid.best_estimator_
print("✔ Grid Search Completed")
print("Best Parameters:", grid.best_params_)


================= STEP 6: GRID SEARCH + CV =================
Fitting 5 folds for each of 432 candidates, totalling 2160 fits
✔ Grid Search Completed
Best Parameters: {'colsample_bytree': 0.8, 'gamma': 0.2, 'learning_rate': 0.05, 'max_depth': 9, 'min_child_weight': 1, 'n_estimators': 600, 'subsample': 0.8}


## 8. FINAL TRAINING WITH EARLY STOPPING

In [8]:
print("\n================= STEP 7: FINAL TRAINING =================")
best_model.set_params(early_stopping_rounds=50)
best_model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test, y_test)],
    verbose=True
)
print("✔ Final training completed")



================= STEP 7: FINAL TRAINING =================
[0]	validation_0-aucpr:0.20974
[1]	validation_0-aucpr:0.23709
[2]	validation_0-aucpr:0.24031
[3]	validation_0-aucpr:0.24685
[4]	validation_0-aucpr:0.24687
[5]	validation_0-aucpr:0.24804
[6]	validation_0-aucpr:0.24714
[7]	validation_0-aucpr:0.24358
[8]	validation_0-aucpr:0.24480
[9]	validation_0-aucpr:0.24648
[10]	validation_0-aucpr:0.24621
[11]	validation_0-aucpr:0.24702
[12]	validation_0-aucpr:0.24724
[13]	validation_0-aucpr:0.24553
[14]	validation_0-aucpr:0.24686
[15]	validation_0-aucpr:0.24723
[16]	validation_0-aucpr:0.24846
[17]	validation_0-aucpr:0.24884
[18]	validation_0-aucpr:0.24846
[19]	validation_0-aucpr:0.24817
[20]	validation_0-aucpr:0.24793
[21]	validation_0-aucpr:0.24853
[22]	validation_0-aucpr:0.24946
[23]	validation_0-aucpr:0.24928
[24]	validation_0-aucpr:0.24975
[25]	validation_0-aucpr:0.25060
[26]	validation_0-aucpr:0.25009
[27]	validation_0-aucpr:0.24985
[28]	validation_0-aucpr:0.25044
[29]	validation_0-aucp

## OPTIONAL CHECKING FOR ACCURACY IMPROVEMENT

In [29]:
# # -----------------------------
# # STEP 8: THRESHOLD OPTIMIZATION
# # -----------------------------
# print("\n================= STEP 8: THRESHOLD OPTIMIZATION =================")
# y_test_prob = best_model.predict_proba(X_test)[:, 1]

# best_thresh = 0.5
# best_f1 = 0
# for t in np.arange(0.1, 0.6, 0.01):
#     y_pred = (y_test_prob >= t).astype(int)
#     f1 = f1_score(y_test, y_pred)
#     if f1 > best_f1:
#         best_f1 = f1
#         best_thresh = t

# print(f"Optimal Threshold Found: {best_thresh:.2f} with F1-score: {best_f1:.2f}")

# y_test_pred = (y_test_prob >= best_thresh).astype(int)


In [31]:
# # -----------------------------
# # STEP 8: ADVANCED THRESHOLD OPTIMIZATION
# # -----------------------------
# print("\n================= STEP 8: ADVANCED THRESHOLD OPTIMIZATION =================")
# y_test_prob = best_model.predict_proba(X_test)[:, 1]

# precision, recall, thresholds = precision_recall_curve(y_test, y_test_prob)
# f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
# best_index = np.argmax(f1_scores)
# optimal_threshold = thresholds[best_index]
# print(f"Optimal Threshold based on PR-curve: {optimal_threshold:.2f}")
# print(f"Maximum F1-score at this threshold: {f1_scores[best_index]:.2f}")

# # Optional tweak to prioritize recall slightly
# tuned_threshold = max(optimal_threshold - 0.05, 0)
# print(f"Tuned Threshold for higher recall: {tuned_threshold:.2f}")

# y_test_pred = (y_test_prob >= tuned_threshold).astype(int)

# # -----------------------------
# # STEP 9: MODEL EVALUATION
# # -----------------------------
# print("\n================= STEP 9: MODEL EVALUATION =================")
# print("\nClassification Report:")
# print(classification_report(y_test, y_test_pred, target_names=['Normal (0)', 'Theft (1)']))
# print("\nConfusion Matrix:")
# print(confusion_matrix(y_test, y_test_pred))


================= STEP 8: ADVANCED THRESHOLD OPTIMIZATION =================
Optimal Threshold based on PR-curve: 0.65
Maximum F1-score at this threshold: 0.32
Tuned Threshold for higher recall: 0.60

================= STEP 9: MODEL EVALUATION =================

Classification Report:
              precision    recall  f1-score   support

  Normal (0)       0.94      0.87      0.90      9689
   Theft (1)       0.24      0.45      0.31       904

    accuracy                           0.83     10593
   macro avg       0.59      0.66      0.61     10593
weighted avg       0.88      0.83      0.85     10593


Confusion Matrix:
[[8410 1279]
 [ 500  404]]


##  9.THRESHOLD OPTIMIZATION

In [1]:
# Baseline optimal threshold from PR-curve
optimal_threshold = 0.60

# High recall tweak
tuned_threshold = 0.45  # lower to boost recall
y_test_pred = (y_test_prob >= tuned_threshold).astype(int)

# Evaluate
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_test_pred, target_names=['Normal (0)', 'Theft (1)']))
print(confusion_matrix(y_test, y_test_pred))

NameError: name 'y_test_prob' is not defined

## OPTIONAL CHECKING FOR ACCURACY IMPROVEMENT

In [24]:
# # -----------------------------
# # STEP 9: MODEL EVALUATION
# # -----------------------------
# print("\n================= STEP 9: MODEL EVALUATION =================")
# print("\nClassification Report:")
# print(classification_report(y_test, y_test_pred, target_names=['Normal (0)', 'Theft (1)']))
# print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))



================= STEP 9: MODEL EVALUATION =================

Classification Report:
              precision    recall  f1-score   support

  Normal (0)       0.94      0.86      0.90      9689
   Theft (1)       0.23      0.45      0.31       904

    accuracy                           0.83     10593
   macro avg       0.59      0.66      0.60     10593
weighted avg       0.88      0.83      0.85     10593

Confusion Matrix:
 [[8348 1341]
 [ 494  410]]


## 10.SAVE MODEL + TOTAL TIME

In [11]:
best_model.save_model("xgb22_theft_model_improved.json")
total_time = time.time() - start_time
print(f"\n✔ Model saved as xgb_theft_model_improved.json")
print(f"⏱ Total Pipeline Time: {total_time/60:.2f} minutes")


✔ Model saved as xgb_theft_model_improved.json
⏱ Total Pipeline Time: 44.79 minutes
